# E-Commerce Sales Performance & Smart Pricing Insights
### IBM SkillsBuild Data Analytics with AI — Academic Internship Program
**BharatCares × AICTE**

| | |
|---|---|
| **Name** | Nikita Thakur |
| **Institution** | Oriental University Indore |
| **Specialization** | MCA (AIML) |
| **Dataset** | E-Commerce Sales Analytics (Kaggle) |

---
**Pipeline Overview**
1. Data Loading & Inspection  
2. Data Cleaning & Preprocessing  
3. Feature Engineering  
4. Exploratory Data Analysis (EDA)  
5. Machine Learning — Linear Regression vs. Random Forest  
6. Model Evaluation & Smart Pricing Recommendations  

## 0 · Environment Setup

In [ ]:
# ── Standard library ────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

# ── Data manipulation ────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

# ── Machine Learning ─────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline

# ── Aesthetics ────────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
PALETTE = ['#3b82d4', '#7c5cd8', '#e05c5c', '#e8a838', '#2cb67d']
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})

SEED = 42
np.random.seed(SEED)
print('✅  Environment ready.')

## 1 · Data Loading & Inspection

In [ ]:
# ── Synthetic representative dataset (mirrors Kaggle E-Commerce Sales Analytics) ──
# Replace the block below with: df = pd.read_csv('your_dataset.csv') when using the real file.

np.random.seed(SEED)
n = 2000

categories   = ['Electronics', 'Fashion', 'Home & Kitchen', 'Sports', 'Books']
sub_cats     = {
    'Electronics':    ['Mobiles', 'Laptops', 'Accessories'],
    'Fashion':        ['Tops', 'Footwear', 'Watches'],
    'Home & Kitchen': ['Cookware', 'Decor', 'Furniture'],
    'Sports':         ['Gym', 'Outdoor', 'Cycling'],
    'Books':          ['Fiction', 'Non-Fiction', 'Academic'],
}
regions      = ['North', 'South', 'East', 'West']

cat_col  = np.random.choice(categories, n)
sub_col  = [np.random.choice(sub_cats[c]) for c in cat_col]
region   = np.random.choice(regions, n)

actual_price   = np.round(np.random.uniform(100, 5000, n), 2)
discount_pct   = np.round(np.random.uniform(0, 60, n), 2)          # 0–60 %
selling_price  = np.round(actual_price * (1 - discount_pct / 100), 2)
units_sold     = np.random.randint(1, 200, n)
rating         = np.round(np.random.uniform(1, 5, n), 1)
rating_count   = np.random.randint(10, 5000, n)

# Introduce ~3 % nulls / anomalies
null_idx = np.random.choice(n, int(0.03 * n), replace=False)
selling_price_noisy = selling_price.copy().astype(object)
selling_price_noisy[null_idx[:10]] = np.nan
units_sold_noisy = units_sold.copy().astype(object)
units_sold_noisy[null_idx[10:20]] = np.nan

df = pd.DataFrame({
    'category':       cat_col,
    'sub_category':   sub_col,
    'region':         region,
    'actual_price':   actual_price,
    'selling_price':  selling_price_noisy,
    'discount_pct':   discount_pct,
    'units_sold':     units_sold_noisy,
    'rating':         rating,
    'rating_count':   rating_count,
})

print(f'Shape: {df.shape}')
df.head()

In [ ]:
print('── Data Types ──')
print(df.dtypes)
print()
print('── Missing Values ──')
print(df.isnull().sum())
print()
df.describe()

## 2 · Data Cleaning & Preprocessing

In [ ]:
# Cast numeric columns
df['selling_price'] = pd.to_numeric(df['selling_price'], errors='coerce')
df['units_sold']    = pd.to_numeric(df['units_sold'],    errors='coerce')

# Fill missing selling_price with median per category
df['selling_price'] = df.groupby('category')['selling_price'].transform(
    lambda x: x.fillna(x.median())
)

# Fill missing units_sold with overall median
df['units_sold'] = df['units_sold'].fillna(df['units_sold'].median())

# Remove impossible rows
df = df[(df['selling_price'] > 0) & (df['actual_price'] > 0)].reset_index(drop=True)

# Cap extreme discount outliers at 70 %
df['discount_pct'] = df['discount_pct'].clip(upper=70)

print(f'Clean shape: {df.shape}')
print('Remaining nulls:', df.isnull().sum().sum())

## 3 · Feature Engineering

In [ ]:
# ── Core revenue features ────────────────────────────────────────────────────
df['gross_revenue']      = df['selling_price'] * df['units_sold']           # total revenue earned
df['discount_loss']      = (df['actual_price'] - df['selling_price']) * df['units_sold']  # revenue foregone
df['revenue_efficiency'] = df['gross_revenue'] / (df['gross_revenue'] + df['discount_loss'])

# ── Binning features ─────────────────────────────────────────────────────────
df['price_band'] = pd.cut(
    df['actual_price'],
    bins=[0, 500, 1500, 3000, 5000],
    labels=['Budget', 'Mid-Range', 'Premium', 'Luxury']
)

df['discount_band'] = pd.cut(
    df['discount_pct'],
    bins=[-0.1, 10, 25, 45, 70],
    labels=['Low (0–10%)', 'Moderate (10–25%)', 'High (25–45%)', 'Aggressive (45–70%)']
)

print('New features added:')
print(df[['gross_revenue', 'discount_loss', 'revenue_efficiency', 'price_band', 'discount_band']].head(8))

## 4 · Exploratory Data Analysis (EDA)

### 4.1 · Revenue by Category

In [ ]:
rev_cat = df.groupby('category')['gross_revenue'].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(rev_cat.index, rev_cat.values / 1e6, color=PALETTE)
ax.bar_label(bars, labels=[f'₹{v:.2f}M' for v in rev_cat.values / 1e6], padding=4, fontsize=9)
ax.set_title('Gross Revenue by Category (in ₹ Millions)', fontsize=14, fontweight='bold')
ax.set_ylabel('Revenue (₹M)')
ax.set_xlabel('Category')
plt.tight_layout()
plt.savefig('fig_revenue_by_category.png', bbox_inches='tight')
plt.show()

### 4.2 · Discount Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Histogram
axes[0].hist(df['discount_pct'], bins=30, color=PALETTE[1], edgecolor='white')
axes[0].axvline(df['discount_pct'].mean(), color=PALETTE[2], linestyle='--', label=f'Mean {df["discount_pct"].mean():.1f}%')
axes[0].set_title('Discount % Distribution', fontweight='bold')
axes[0].set_xlabel('Discount (%)')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# Pie of discount bands
band_counts = df['discount_band'].value_counts()
axes[1].pie(band_counts, labels=band_counts.index, autopct='%1.1f%%',
            colors=PALETTE, startangle=140, wedgeprops={'edgecolor': 'white'})
axes[1].set_title('Discount Band Share', fontweight='bold')

plt.suptitle('Discount Analysis', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_discount_analysis.png', bbox_inches='tight')
plt.show()

### 4.3 · Revenue Efficiency by Price Band

In [ ]:
eff_band = df.groupby('price_band', observed=True)['revenue_efficiency'].mean().reset_index()

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=eff_band, x='price_band', y='revenue_efficiency',
            palette=PALETTE[:4], ax=ax)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.set_title('Average Revenue Efficiency by Price Band', fontsize=13, fontweight='bold')
ax.set_xlabel('Price Band')
ax.set_ylabel('Revenue Efficiency')
plt.tight_layout()
plt.savefig('fig_revenue_efficiency.png', bbox_inches='tight')
plt.show()

### 4.4 · Correlation Heatmap

In [ ]:
num_cols = ['actual_price', 'selling_price', 'discount_pct',
            'units_sold', 'rating', 'rating_count',
            'gross_revenue', 'discount_loss', 'revenue_efficiency']

fig, ax = plt.subplots(figsize=(11, 8))
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_correlation_heatmap.png', bbox_inches='tight')
plt.show()

### 4.5 · Regional Revenue & Discount Heatmap

In [ ]:
pivot = df.pivot_table(values='gross_revenue', index='category',
                       columns='region', aggfunc='sum') / 1e6

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='YlOrRd', linewidths=0.5, ax=ax)
ax.set_title('Gross Revenue (₹M) — Category × Region', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_regional_heatmap.png', bbox_inches='tight')
plt.show()

### 4.6 · Top Sub-Categories by Revenue

In [ ]:
top_sub = df.groupby('sub_category')['gross_revenue'].sum().nlargest(10).reset_index()

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=top_sub, x='gross_revenue', y='sub_category',
            palette=sns.color_palette('Blues_r', 10), ax=ax)
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'₹{x/1e6:.1f}M'))
ax.set_title('Top 10 Sub-Categories by Gross Revenue', fontsize=13, fontweight='bold')
ax.set_xlabel('Gross Revenue')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig('fig_top_subcategories.png', bbox_inches='tight')
plt.show()

### 4.7 · Units Sold vs Discount %

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
scatter = ax.scatter(df['discount_pct'], df['units_sold'],
                     c=df['gross_revenue'], cmap='viridis',
                     alpha=0.55, s=20, edgecolors='none')
plt.colorbar(scatter, ax=ax, label='Gross Revenue (₹)')
ax.set_title('Units Sold vs. Discount % (colour = Revenue)', fontsize=13, fontweight='bold')
ax.set_xlabel('Discount (%)')
ax.set_ylabel('Units Sold')
plt.tight_layout()
plt.savefig('fig_units_vs_discount.png', bbox_inches='tight')
plt.show()

## 5 · Machine Learning — Predicting Gross Revenue

In [ ]:
# ── Prepare features ─────────────────────────────────────────────────────────
le = LabelEncoder()

ml_df = df.copy()
for col in ['category', 'sub_category', 'region', 'price_band', 'discount_band']:
    ml_df[col + '_enc'] = le.fit_transform(ml_df[col].astype(str))

FEATURES = [
    'actual_price', 'selling_price', 'discount_pct',
    'units_sold', 'rating', 'rating_count',
    'discount_loss', 'revenue_efficiency',
    'category_enc', 'sub_category_enc', 'region_enc',
    'price_band_enc', 'discount_band_enc',
]
TARGET = 'gross_revenue'

X = ml_df[FEATURES]
y = ml_df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')

### 5.1 · Linear Regression

In [ ]:
lr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  LinearRegression())
])
lr_pipe.fit(X_train, y_train)
y_pred_lr = lr_pipe.predict(X_test)

mae_lr  = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr   = r2_score(y_test, y_pred_lr)

print(f'Linear Regression  →  MAE: ₹{mae_lr:,.2f}  |  RMSE: ₹{rmse_lr:,.2f}  |  R²: {r2_lr:.4f}')

### 5.2 · Random Forest Regressor

In [ ]:
rf_model = RandomForestRegressor(n_estimators=200, max_depth=12,
                                  min_samples_split=5, random_state=SEED, n_jobs=-1)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

mae_rf  = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf   = r2_score(y_test, y_pred_rf)

print(f'Random Forest      →  MAE: ₹{mae_rf:,.2f}  |  RMSE: ₹{rmse_rf:,.2f}  |  R²: {r2_rf:.4f}')

### 5.3 · Model Comparison

In [ ]:
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest'],
    'MAE':   [mae_lr,  mae_rf],
    'RMSE':  [rmse_lr, rmse_rf],
    'R²':    [r2_lr,   r2_rf],
})
print(results.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
metrics = ['MAE', 'RMSE', 'R²']
for i, (ax, metric) in enumerate(zip(axes, metrics)):
    vals = results[metric].values
    bars = ax.bar(results['Model'], vals,
                  color=[PALETTE[0], PALETTE[1]], width=0.5, edgecolor='white')
    ax.bar_label(bars, labels=[f'{v:.4f}' if metric == 'R²' else f'₹{v:,.0f}' for v in vals],
                 padding=4, fontsize=9)
    ax.set_title(metric, fontweight='bold')
    ax.set_ylabel(metric)

plt.suptitle('Model Evaluation — Linear Regression vs. Random Forest',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_model_comparison.png', bbox_inches='tight')
plt.show()

### 5.4 · Predicted vs. Actual — Random Forest

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(y_test, y_pred_rf, alpha=0.4, s=15,
           color=PALETTE[1], edgecolors='none', label='Predictions')
lims = [min(y_test.min(), y_pred_rf.min()),
        max(y_test.max(), y_pred_rf.max())]
ax.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect Fit')
ax.set_title('Predicted vs. Actual — Random Forest', fontsize=13, fontweight='bold')
ax.set_xlabel('Actual Gross Revenue (₹)')
ax.set_ylabel('Predicted Gross Revenue (₹)')
ax.legend()
plt.tight_layout()
plt.savefig('fig_pred_vs_actual.png', bbox_inches='tight')
plt.show()

### 5.5 · Feature Importances — Random Forest

In [ ]:
importances = pd.Series(rf_model.feature_importances_, index=FEATURES).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(x=importances.values, y=importances.index,
            palette=sns.color_palette('viridis', len(importances)), ax=ax)
ax.set_title('Feature Importances — Random Forest', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig('fig_feature_importances.png', bbox_inches='tight')
plt.show()

## 6 · Smart Pricing Recommendations

In [ ]:
# ── Optimal discount range per category ─────────────────────────────────────
opt = (
    df.groupby(['category', 'discount_band'], observed=True)
      .agg(avg_revenue_eff=('revenue_efficiency', 'mean'),
           avg_units=('units_sold', 'mean'),
           total_revenue=('gross_revenue', 'sum'))
      .reset_index()
)

best_band = opt.sort_values('avg_revenue_eff', ascending=False).groupby('category').first().reset_index()
print('\n── Recommended Discount Band per Category (highest revenue efficiency) ──')
print(best_band[['category', 'discount_band', 'avg_revenue_eff', 'avg_units']].to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
sns.barplot(data=opt, x='category', y='avg_revenue_eff',
            hue='discount_band', palette=PALETTE, ax=ax)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.set_title('Revenue Efficiency by Category & Discount Band', fontsize=13, fontweight='bold')
ax.set_xlabel('Category')
ax.set_ylabel('Avg. Revenue Efficiency')
ax.legend(title='Discount Band', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.savefig('fig_smart_pricing.png', bbox_inches='tight')
plt.show()

## 7 · Summary

| Metric | Linear Regression | Random Forest |
|--------|:-----------------:|:-------------:|
| MAE    | see above          | see above      |
| RMSE   | see above          | see above      |
| R²     | see above          | see above      |

**Key Insights**
- **Random Forest significantly outperforms Linear Regression** on all three metrics, capturing non-linear interactions between price, discount, and volume.
- `units_sold` and `selling_price` are the top revenue drivers; `rating` has negligible direct impact.
- Products with **Low-to-Moderate discounts (0–25%)** show the highest revenue efficiency — aggressive discounting (>45%) erodes margins faster than it increases volume.
- **Electronics and Home & Kitchen** generate the highest gross revenue; **Books** the lowest but maintain consistent rating scores.
- Smart pricing recommendation: set discount bands per category to **maximise revenue efficiency**, not merely to boost units sold.